In [ ]:
import torch
import torch.nn as nn
from torch import Tensor
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader,SubsetRandomSampler,ConcatDataset
import torch.nn.functional as F

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

from model import SeqNN

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
# Load the entire model (architecture + weights)
model = torch.load("/home1/smaruj/pytorch_akita/model.pth")

In [ ]:
model = model.to(device)

In [ ]:
# Set the model to evaluation mode (important for inference)
model.eval()

In [ ]:
from torchinfo import summary

summary(model, input_size=(2, 4, 1048576), col_names=["output_size", "num_params"])

In [ ]:
# from pyfaidx import Fasta

In [ ]:
# fasta_file = "/project/fudenber_735/genomes/hg38/hg38.fa"
# genome = Fasta(fasta_file)

# region = "chr12"
# start = 115163136
# end = 116211712

# region = "chr11"
# start = 75429888
# end = 76478464

# region = "chr15"
# start = 63281152
# end = 64329728

In [ ]:
# sequence = genome[region][start:end]

In [ ]:
import numpy as np

In [ ]:
def one_hot_encode_sequence(sequence_obj):
    # Convert pyfaidx.Sequence object to string
    sequence = str(sequence_obj).upper()

    # Define the mapping from bases to integers
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    # Step 1: Encode the sequence to integers
    encoded_sequence = np.array([base_to_int[base] for base in sequence])

    # Step 2: One-hot encode the sequence
    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    # Step 3: Expand the dimensions to [1, 4, sequence_length]
    input_sequence = np.expand_dims(one_hot_encoded, axis=0)

    return input_sequence

In [ ]:
background_file = f"/project/fudenber_735/akitaX1_analyses_data/background_generation/background_generation/background_sequences_model_0.fa"

In [ ]:
background_seqs = []
    
with open(background_file, "r") as f:
    for line in f.readlines():
        if ">" in line:
            continue
        background_seqs.append(one_hot_encode_sequence(line.strip()[:1048576]))

In [ ]:
X = background_seqs[0]

In [ ]:
# Convert the NumPy array to a PyTorch tensor
X_tensor = torch.tensor(X).to("cuda:0")

In [ ]:
# torch.save(X_tensor, "/scratch1/smaruj/ledidi_targets/constant_boundary_background/X_0.pt")

In [ ]:
model.eval()
with torch.no_grad():
    y = model(X_tensor, training=False)

In [ ]:
# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value

def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [ ]:
target0 = y[:,0,:]

In [ ]:
matrix = from_upper_triu(target0, matrix_len=448, num_diags=2)

In [ ]:
import matplotlib.pyplot as plt

# Plot the matrix
plt.figure(figsize=(8, 8))
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-2, vmax=2)
plt.colorbar()
plt.title('Heatmap of the Matrix')
plt.show()

In [ ]:
size = 448
c = -1.25
boundary = np.zeros((size, size))
half = size // 2

# creating a boundary
boundary[:half, half:] = c
boundary[half:, :half] = c

In [ ]:
plt.figure(figsize=(8, 8))
plt.matshow(boundary.astype(np.float16), cmap='RdBu_r', vmin=-2, vmax=2)
plt.colorbar()
plt.title('Constant Boundary')
plt.show()

In [ ]:
def upper_triangular_to_vector_skip_two_diagonals(matrix):
    """
    Extracts the upper triangular part of a square matrix (excluding the first two diagonals) 
    and transforms it into a vector.
    
    Parameters:
        matrix (np.ndarray): A 2D numpy matrix of shape (448, 448).
        
    Returns:
        np.ndarray: A 1D array containing the upper triangular elements (excluding the first two diagonals).
    """
    if matrix.shape != (448, 448):
        raise ValueError("Input matrix must be of shape (448, 448).")
    
    # Extract the upper triangular part excluding the first two diagonals
    upper_triangular_vector = matrix[np.triu_indices(448, k=2)]
    
    return upper_triangular_vector

In [ ]:
strong_up_triu = upper_triangular_to_vector_skip_two_diagonals(boundary)

In [ ]:
y_cpu = y.cpu()

In [ ]:
y_bar_cpu = y_cpu.clone().detach()

y_bar_cpu[:, :5, :] = y_cpu[:, :5, :] + strong_up_triu

In [ ]:
goal = matrix + boundary

In [ ]:
plt.figure(figsize=(8, 8))
plt.matshow(goal.astype(np.float16), cmap='RdBu_r', vmin=-2, vmax=2)
plt.colorbar()
plt.title('Desired Matrix')
plt.show()

In [ ]:
modified_vector = torch.tensor(y_bar_cpu)

In [ ]:
modified_vector_tensor = modified_vector.to(device)

In [ ]:
torch.save(modified_vector_tensor, "/scratch1/smaruj/ledidi_targets/constant_boundary_background/modified_y_1.25.pt")

In [ ]:
def store_tower_output(ohe_sequence, model):
    x = model.conv_block_1(ohe_sequence)
    x = model.conv_tower(x)
    # save the tensor
    print(x.shape)
    torch.save(x, "/scratch1/smaruj/ledidi_targets/constant_boundary_background/tower_out.pt")
    torch.cuda.empty_cache()

In [ ]:
store_tower_output(X_tensor, model)